# UD6.04. La capa de confianza: resúmenes, cadenas y cuándo no hacen falta

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloque 7 de los apuntes · Criterios **3.c** y **3.d**

---

La cadena de bloques es la tecnología de esta unidad que **más se propone y menos se
justifica**. Aparece en la mitad de los trabajos de convergencia tecnológica, casi siempre
como «se añadirá blockchain para garantizar la trazabilidad», y casi nunca con una frase
detrás que diga entre quiénes falta la confianza.

Este cuaderno la construye entera —son cuarenta líneas— para que se vea que no tiene nada
mágico dentro, y después construye **el contraejemplo**, que es lo que de verdad hay que
saber: una tabla firmada que hace lo mismo, más deprisa y sin ninguna red.

Al final tienes que poder contestar la pregunta que separa un análisis de un folleto:

> **¿Entre quiénes falta la confianza en este sistema?**

Si la respuesta es «entre nadie, hay un administrador», la cadena de bloques sobra.


In [ ]:
import hashlib
import hmac
import os
import time
from dataclasses import dataclass

SEMILLA = 20262027


---

## 1. El resumen criptográfico

La pieza de abajo del todo. Una función que convierte cualquier cantidad de datos en una
cadena de longitud fija con tres propiedades:

1. **Determinista**: la misma entrada da siempre la misma salida.
2. **Efecto avalancha**: cambiar un bit cambia media salida.
3. **Irreversible en la práctica**: de la salida no se vuelve a la entrada.

La segunda es la que hace útil todo lo demás, y se ve mejor contándola que leyéndola.


In [ ]:
def resumen(datos):
    '''SHA-256 en hexadecimal. Acepta texto o bytes.'''
    if isinstance(datos, str):
        datos = datos.encode("utf-8")
    return hashlib.sha256(datos).hexdigest()


a = "temperatura=21.5 sensor=17 hora=09:41:02"
b = "temperatura=21.6 sensor=17 hora=09:41:02"

print(f"{a}\n  -> {resumen(a)}")
print(f"{b}\n  -> {resumen(b)}")
print()

# Cuántos BITS cambian al cambiar un solo carácter de la entrada.
bits_a = bin(int(resumen(a), 16))[2:].zfill(256)
bits_b = bin(int(resumen(b), 16))[2:].zfill(256)
distintos = sum(1 for x, y in zip(bits_a, bits_b) if x != y)

print(f"Han cambiado {distintos} bits de 256, un {distintos / 256:.1%}.")
print("Lo esperable es la mitad, y eso es el efecto avalancha: un decimal de más en")
print("una lectura da un resumen que no se parece en nada al anterior.")
print()
print("Por eso el resumen sirve de huella: no hace falta guardar el dato para saber")
print("si el dato ha cambiado.")


### Lo que cuesta, que es la pregunta del criterio 3.c

«Registrar la procedencia de los datos» suena caro. Vamos a medirlo, porque la sorpresa es
que la parte criptográfica **no cuesta nada**, y eso cambia dónde está el problema de verdad.


In [ ]:
for mb in [1, 10, 50, 200]:
    datos = os.urandom(mb * 1024 * 1024)
    t = time.perf_counter()
    huella = hashlib.sha256(datos).hexdigest()
    ms = (time.perf_counter() - t) * 1000
    print(f"{mb:>4} MB  ->  {ms:>7.1f} ms   ({mb / (ms / 1000):>7.0f} MB/s)   "
          f"{huella[:16]}...")

print()
print("Del orden de mil megabytes por segundo. Calcular la huella de un conjunto de")
print("entrenamiento entero cuesta menos que abrirlo.")
print()
print("Conclusión que hay que escribir en A6.2: el coste de la trazabilidad NO está en")
print("la criptografía. Está en montar y mantener la infraestructura que hace que ese")
print("resumen sea creíble para alguien de fuera. Confundir las dos cosas es el error")
print("más común al evaluar esta tecnología.")


---

## 2. El encadenamiento

Segunda pieza. Cada bloque guarda el resumen del anterior, así que el resumen del último
depende de **todos** los contenidos anteriores. Eso es toda la cadena de bloques: el resto
son detalles de reparto.


In [ ]:
@dataclass
class Bloque:
    indice: int
    anterior: str
    contenido: str

    def calcula(self):
        return resumen(f"{self.indice}|{self.anterior}|{self.contenido}")


def construye_cadena(contenidos):
    bloques = []
    anterior = "0" * 64          # el bloque cero no tiene padre: se usa un resumen nulo
    for i, contenido in enumerate(contenidos):
        b = Bloque(i, anterior, contenido)
        b.resumen = b.calcula()
        bloques.append(b)
        anterior = b.resumen
    return bloques


def valida(bloques):
    '''Devuelve (es_valida, indice_del_primer_bloque_incoherente).'''
    anterior = "0" * 64
    for b in bloques:
        if b.anterior != anterior or b.calcula() != b.resumen:
            return False, b.indice
        anterior = b.resumen
    return True, -1


# Un registro de decisiones de un modelo de concesión de crédito. Es el caso donde la
# trazabilidad tiene sentido de verdad: sector regulado, decisiones sobre personas.
REGISTROS = [
    "modelo=credito-v1.3 conjunto=sha256:9f2a... cliente=A-1042 decision=aprobado",
    "modelo=credito-v1.3 conjunto=sha256:9f2a... cliente=A-1043 decision=denegado",
    "modelo=credito-v1.3 conjunto=sha256:9f2a... cliente=A-1044 decision=aprobado",
    "modelo=credito-v1.4 conjunto=sha256:c81b... cliente=A-1045 decision=denegado",
    "modelo=credito-v1.4 conjunto=sha256:c81b... cliente=A-1046 decision=aprobado",
    "modelo=credito-v1.4 conjunto=sha256:c81b... cliente=A-1047 decision=denegado",
]

cadena = construye_cadena(REGISTROS)
for b in cadena:
    print(f"  bloque {b.indice}   anterior {b.anterior[:10]}...   "
          f"propio {b.resumen[:10]}...   {b.contenido[-28:]}")

ok, _ = valida(cadena)
print(f"\ncadena válida: {ok}")


### La manipulación

Ahora lo interesante. Alguien quiere cambiar una decisión denegada por una aprobada, y **no
es tonto**: recalcula el resumen del bloque que toca, para que el bloque quede coherente
consigo mismo.

Mira dónde salta la incoherencia.


In [ ]:
victima = 3
antes = cadena[victima].contenido
cadena[victima].contenido = antes.replace("denegado", "aprobado")

# Esta comprobación está puesta a propósito: la primera versión de este cuaderno
# «manipulaba» un bloque que ya decía aprobado, el reemplazo no cambiaba nada, la
# cadena seguía siendo válida y el ejemplo demostraba lo contrario de lo que quería.
assert cadena[victima].contenido != antes, "la manipulación no ha cambiado nada"

cadena[victima].resumen = cadena[victima].calcula()      # rehace SU propio resumen

ok, donde = valida(cadena)
print(f"antes:   {antes[-28:]}")
print(f"después: {cadena[victima].contenido[-28:]}")
print()
print(f"cadena válida: {ok}    primer bloque incoherente: {donde}")
print()
print(f"El bloque {victima} es coherente consigo mismo: su resumen cuadra con su nuevo")
print(f"contenido. Lo que no cuadra es el bloque {donde}, que guarda el resumen VIEJO")
print(f"del {victima} en su campo «anterior».")
print()
print("Para ocultar el cambio habría que rehacer desde el bloque manipulado hasta el")
print("final. Y ese es todo el mecanismo, sin nada de magia:")
print()
print("    el coste de mentir crece con lo que venga detrás.")


In [ ]:
# Cuánto costaría rehacer la cola de la cadena. Con un registro por decisión y un
# volumen realista, la respuesta es incómoda.
DECISIONES_DIA = 5_000

t = time.perf_counter()
for _ in range(20_000):
    resumen("un registro de ejemplo de longitud parecida a los de arriba")
por_resumen = (time.perf_counter() - t) / 20_000

print(f"un resumen cuesta {por_resumen * 1e6:.2f} microsegundos")
for dias in [1, 30, 365]:
    n = DECISIONES_DIA * dias
    print(f"  rehacer {dias:>3} día(s) de cadena ({n:>9,} bloques): "
          f"{n * por_resumen:>8.2f} s")

print()
print("Un año de cadena se rehace en segundos. Encadenar POR SÍ SOLO no protege de")
print("nada a quien tenga el fichero: protege cuando MUCHOS nodos independientes")
print("guardan su copia y hay que convencer a la mayoría.")
print()
print("Esa es la pieza cara, y es la única razón para usar una cadena de bloques de")
print("verdad en lugar de las cuarenta líneas de arriba.")


---

## 3. El contraejemplo, que es lo que hay que saber

Ahora la parte que casi nunca se cuenta. **Una tabla normal con cada fila firmada detecta la
manipulación exactamente igual de bien.** Se firma con HMAC, que es un resumen con clave: sin
la clave no se puede fabricar una firma válida.

Vamos a hacerlo, atacarlo igual y comparar.


In [ ]:
CLAVE = os.urandom(32)          # la guarda quien administra el sistema


def firma(registro, clave=CLAVE):
    return hmac.new(clave, registro.encode("utf-8"), hashlib.sha256).hexdigest()


tabla = [{"id": i, "registro": r, "firma": firma(r)} for i, r in enumerate(REGISTROS)]

t = time.perf_counter()
validas = all(hmac.compare_digest(fila["firma"], firma(fila["registro"]))
              for fila in tabla)
us = (time.perf_counter() - t) * 1e6
print(f"validar las {len(tabla)} filas: {us:.0f} microsegundos   todas válidas: {validas}")

# El mismo ataque: cambiar el contenido de una fila.
tabla[3]["registro"] = tabla[3]["registro"].replace("denegado", "aprobado")
malas = [f["id"] for f in tabla if not hmac.compare_digest(f["firma"], firma(f["registro"]))]
print(f"tras manipular la fila 3, filas que no validan: {malas}")

# Y el ataque de verdad: intentar rehacer la firma SIN la clave.
falsificada = hmac.new(os.urandom(32), tabla[3]["registro"].encode(), hashlib.sha256)
print(f"firma fabricada con otra clave: ¿cuela? "
      f"{hmac.compare_digest(falsificada.hexdigest(), firma(tabla[3]['registro']))}")
print()
print("No cuela. Sin la clave no se puede fabricar una firma válida, igual que sin")
print("rehacer la cola no se puede ocultar un cambio en la cadena.")


### Entonces, ¿en qué se diferencian?

En una sola cosa, y no es técnica:

| | Tabla firmada | Cadena de bloques |
|---|---|---|
| Detecta manipulación | Sí | Sí |
| Coste de validar | Microsegundos | Microsegundos |
| Infraestructura | Ninguna | Muchos nodos independientes |
| **De quién protege** | **De todos menos del que tiene la clave** | **También del que administra** |

Esa última fila es la pregunta entera. Si hay **un administrador de confianza** —y en la
inmensa mayoría de los sistemas de una empresa lo hay—, la tabla firmada hace el trabajo y es
más simple, más barata y más rápida.

La cadena de bloques resuelve el caso en el que **las partes no se fían entre sí** y no hay
nadie a quien todas acepten como árbitro.


In [ ]:
CASOS = [
    ("Registro de decisiones de un modelo interno, para auditoría propia",
     "un administrador de confianza", False),
    ("Trazabilidad de la cadena de frío entre productor, transportista y tienda",
     "tres empresas que compiten y se reclaman daños", True),
    ("Huella del conjunto de entrenamiento, para reproducir experimentos",
     "el propio equipo de datos", False),
    ("Prueba ante un regulador de qué modelo decidió qué, cuando la empresa es parte",
     "empresa y regulador, con la empresa custodiando la prueba", True),
    ("Consorcio de hospitales que entrena un modelo común y reparte mérito",
     "hospitales independientes sin árbitro común", True),
    ("Registro de temperaturas de una cámara, para el propio mantenimiento",
     "un administrador de confianza", False),
]

print(f"{'caso':<68}{'¿cadena?':>10}")
print("-" * 78)
for caso, quien, justifica in CASOS:
    print(f"{caso:<68}{'sí' if justifica else 'no':>10}")
    print(f"    entre quiénes falta la confianza: {quien}")

justificados = sum(1 for _, _, j in CASOS if j)
print()
print(f"{justificados} de {len(CASOS)}. Y fíjate en el patrón: los tres que la justifican")
print("tienen varias organizaciones con intereses enfrentados. Los tres que no, tienen")
print("un dueño único del sistema.")
print()
print("Regla para la actividad y para PR6:")
print("  una propuesta de cadena de bloques que no nombre a las partes que no se fían")
print("  entre sí NO está justificada, y se corrige como no justificada.")


---

## 4. Lo que sí aporta siempre: la huella del modelo y del conjunto

Independientemente de si hace falta cadena de bloques, hay una práctica de esta unidad que
cuesta cuatro líneas y que deberías llevarte a cualquier proyecto: **registrar la huella de
lo que se usó**.

Es la respuesta a una pregunta que aparece siempre y que casi nunca se puede contestar: *este
modelo que está en producción, ¿con qué datos se entrenó exactamente?*


In [ ]:
def huella_de_fichero(ruta, bloque=1024 * 1024):
    '''Resumen de un fichero leído a trozos, para que quepa cualquier tamaño.'''
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()


# Se fabrica un conjunto de ejemplo para poder medir de verdad.
import tempfile

carpeta = tempfile.mkdtemp()
ruta = os.path.join(carpeta, "conjunto.csv")
with open(ruta, "w", encoding="utf-8") as f:
    f.write("sensor,hora,temperatura\n")
    for i in range(200_000):
        f.write(f"{i % 40},{i},{20 + (i % 100) / 10:.1f}\n")

tamano = os.path.getsize(ruta)
t = time.perf_counter()
h = huella_de_fichero(ruta)
ms = (time.perf_counter() - t) * 1000

ficha = {
    "modelo": "deteccion-anomalias-v2.1",
    "conjunto": os.path.basename(ruta),
    "bytes": tamano,
    "sha256": h,
    "filas": 200_000,
}
print(f"conjunto de {tamano / 1e6:.1f} MB, huella calculada en {ms:.0f} ms")
print()
for clave, valor in ficha.items():
    print(f"  {clave:<10}{valor}")

print()
print("Esas cinco líneas guardadas junto al modelo contestan para siempre la pregunta")
print("de con qué se entrenó. No necesitan red, ni cadena, ni permiso de nadie.")
print()
print("Es el 90 % del valor de la trazabilidad por el 0 % del coste, y va en la ficha")
print("del modelo que la PR5 de la UD5 ya pedía. Lo que añade la cadena de bloques es")
print("solo que un tercero se lo crea sin fiarse de ti.")


---

## 5. Los contratos inteligentes, en una pantalla

Un contrato inteligente es un programa que se ejecuta en la propia cadena cuando se cumple
una condición registrada en ella. El ejemplo típico de esta unidad: **si la cadena de frío se
rompió, se paga la indemnización automáticamente**.

Lo interesante no es el programa, que es trivial. Es el agujero que tiene debajo.


In [ ]:
def contrato_cadena_de_frio(lecturas, umbral=8.0, minutos_tolerados=30):
    '''Paga si la temperatura pasó del umbral más tiempo del tolerado.
    Solo puede mirar lo que esté REGISTRADO: esa es su virtud y su límite.'''
    seguidos = 0
    peor = 0
    for _, grados in lecturas:
        seguidos = seguidos + 1 if grados > umbral else 0
        peor = max(peor, seguidos)
    return {"minutos_seguidos": peor, "paga": peor > minutos_tolerados}


buena = [(i, 4.0 + (i % 5) * 0.2) for i in range(600)]
mala = [(i, 4.0 if not 200 <= i < 260 else 11.5) for i in range(600)]

for nombre, lecturas in [("cadena de frío correcta", buena), ("con una rotura", mala)]:
    r = contrato_cadena_de_frio(lecturas)
    print(f"{nombre:<26} peor racha {r['minutos_seguidos']:>4} min   "
          f"paga: {r['paga']}")

print()
print("El programa funciona. El problema está en la primera línea de la función:")
print("«solo puede mirar lo que esté registrado».")
print()
print("Alguien tiene que poner ahí el dato del mundo físico, y ese alguien es un")
print("intermediario en quien hay que confiar: el sensor, quien lo calibró, quien")
print("lo instaló, y quien publica sus lecturas en la cadena.")
print()
print("La convergencia no elimina la confianza. La DESPLAZA, y saber adónde la")
print("desplaza es exactamente el criterio 3.d: «se ha evaluado cómo la convergencia")
print("tecnológica aporta seguridad en los negocios».")


---

## Lo que hay que llevarse

1. **El resumen criptográfico no cuesta nada**: del orden de mil megabytes por segundo. El
   coste de la trazabilidad no está en la criptografía.
2. **Encadenar es guardar el resumen del anterior**, y son cuarenta líneas. El bloque
   manipulado queda coherente; el siguiente, no.
3. **Encadenar por sí solo no protege de quien tenga el fichero**: rehacer un año de cadena
   son segundos. Lo que protege es que muchos nodos independientes guarden copia, y eso es lo
   caro.
4. **Una tabla firmada con HMAC detecta la manipulación igual de bien**, en microsegundos y
   sin red. La diferencia es de quién protege, no de si protege.
5. **La cadena de bloques resuelve falta de confianza entre partes.** Una propuesta que no
   nombre a esas partes no está justificada.
6. **La huella del conjunto y del modelo, en la ficha, cuesta cuatro líneas** y contesta para
   siempre con qué se entrenó. Llévatela a todos tus proyectos.
7. **Un contrato inteligente solo ve lo que está registrado**, así que alguien tiene que
   registrar el mundo físico. La confianza no desaparece: se desplaza.
